<a id="activity"></a>
# VideoDB Understanding: Activity

Turn a video into a timeline of actions and goals with the `activity` analyzer, then play any moment from it.

The sample is a first-person video of a person folding laundry, uploaded straight from its URL.

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/guides/indexing-v2/understanding/activity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a id="install"></a>
## 1. Install and connect

Enter your VideoDB API key when prompted, or set `VIDEO_DB_API_KEY` in your environment or `.env`.

In [ ]:
!pip install -q "videodb>=0.5.0" python-dotenv pandas

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from videodb import connect

load_dotenv()
if not os.getenv("VIDEO_DB_API_KEY"):
    os.environ["VIDEO_DB_API_KEY"] = getpass("Enter your VideoDB API key: ")

conn = connect(api_key=os.environ["VIDEO_DB_API_KEY"])
collection = conn.get_collection()
print("Connected to VideoDB")
print("Collection:", collection.id)

<a id="helpers"></a>
## 2. Helpers
Everything that renders or plays results lives here, so the rest of the notebook only calls it. The code is collapsed by default; run the three cells and move on.
Everything that renders or plays results lives here, so the rest of the notebook only calls it.

| Helper | What it does |
|---|---|
| `activity_tables(output)` | Splits the analyzer output into `actions`, `goals` and `warnings` |
| `show_summary(...)` | One-line overview plus any processing warnings |
| `show_timeline(actions)` | Visual timeline, one lane per goal |
| `show_actions(actions)` / `show_goals(goals)` | Readable tables |
| `play_action(...)` / `play_goal(...)` | Stream just that moment of the video |

In [ ]:
#@title Data helpers: activity_tables (run this cell; double-click to view the code)
from html import escape

import pandas as pd
from IPython.display import HTML, display
from videodb import play_stream


def clock(seconds):
    """Format seconds as m:ss."""
    if pd.isna(seconds):
        return "–"
    minutes, secs = divmod(int(round(seconds)), 60)
    return f"{minutes}:{secs:02d}"


def activity_tables(output):
    """Split Activity output into (actions, goals, warnings)."""
    metadata = output.get("metadata") or {}
    rows = []
    for scene in output.get("scenes") or []:
        data = scene.get("data") or {}
        rows.append({
            "scene_id": scene.get("scene_id"),
            "start": scene.get("start"),
            "end": scene.get("end"),
            "action": data.get("instruction"),
            "goal": data.get("goal_instruction"),
            "confidence": data.get("confidence"),
            "issue": data.get("issue"),
        })
    actions = pd.DataFrame(
        rows, columns=["scene_id", "start", "end", "action", "goal", "confidence", "issue"]
    )
    goals = pd.DataFrame(metadata.get("goals") or [])
    return actions, goals, metadata.get("warnings") or []

In [ ]:
#@title Display helpers: summary, timeline, tables (run this cell; double-click to view the code)
GOAL_COLORS = ["#4C78A8", "#F58518", "#54A24B", "#B279A2", "#E45756", "#72B7B2", "#EECA3B", "#9D755D"]


def show_summary(actions, goals, warnings):
    """Show what the run produced and anything that needs review."""
    timed = actions.dropna(subset=["start", "end"])
    span = f"{clock(timed['start'].min())}–{clock(timed['end'].max())}" if len(timed) else "no timed actions"
    flagged = int(actions["issue"].notna().sum())
    parts = [f"<b>{len(actions)}</b> actions", f"<b>{len(goals)}</b> goals", span]
    if flagged:
        parts.append(f"<b>{flagged}</b> with an issue")
    html = f"<p style='font-size:15px'>{' &nbsp;·&nbsp; '.join(parts)}</p>"
    if warnings:
        items = "".join(f"<li>{escape(str(warning))}</li>" for warning in warnings)
        html += f"<p><b>Processing warnings</b>: review before relying on the timeline.</p><ul>{items}</ul>"
    display(HTML(html))


def show_timeline(actions):
    """Draw the actions on a time axis, one lane per goal. Hover a block for its action."""
    timed = actions.dropna(subset=["start", "end"])
    if timed.empty:
        print("No timed actions to draw.")
        return
    total = float(timed["end"].max()) or 1.0
    lanes = []
    for lane, (goal, group) in enumerate(timed.groupby(timed["goal"].fillna("No goal"), sort=False)):
        blocks = ""
        for row, action in group.iterrows():
            left = 100 * action["start"] / total
            width = max(100 * (action["end"] - action["start"]) / total, 0.4)
            tip = f"#{row} {clock(action['start'])}–{clock(action['end'])} {action['action'] or ''}"
            outline = "outline:2px dashed #d62728;" if pd.notna(action["issue"]) else ""
            blocks += (
                f"<div title='{escape(tip, quote=True)}' style='position:absolute;left:{left:.2f}%;"
                f"width:{width:.2f}%;top:3px;bottom:3px;border-radius:3px;"
                f"background:{GOAL_COLORS[lane % len(GOAL_COLORS)]};{outline}'></div>"
            )
        lanes.append(
            "<div style='display:flex;align-items:center;margin:3px 0'>"
            "<div style='width:30%;padding-right:10px;font-size:12px;overflow:hidden;"
            f"white-space:nowrap;text-overflow:ellipsis' title='{escape(goal, quote=True)}'>{escape(goal)}</div>"
            "<div style='position:relative;flex:1;height:22px;border-radius:3px;"
            f"background:rgba(128,128,128,.15)'>{blocks}</div></div>"
        )
    ticks = "".join(f"<span>{clock(total * step / 4)}</span>" for step in range(5))
    axis = (
        "<div style='display:flex;margin-top:2px'><div style='width:30%'></div>"
        f"<div style='flex:1;display:flex;justify-content:space-between;font-size:11px;opacity:.7'>{ticks}</div></div>"
    )
    display(HTML(f"<div style='max-width:900px'>{''.join(lanes)}{axis}</div>"))


def show_actions(actions):
    """Show the action table. The row number is what `play_action` takes."""
    table = actions.assign(start=actions["start"].map(clock), end=actions["end"].map(clock))
    table = table.drop(columns=["scene_id"]).fillna({"issue": ""})
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
        display(table)


def show_goals(goals):
    """Show the goal table. The row number is what `play_goal` takes."""
    columns = [c for c in ["goal_id", "instruction", "start", "end", "uncertainty"] if c in goals]
    table = goals[columns].copy()
    for column in ("start", "end"):
        if column in table:
            table[column] = table[column].map(clock)
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
        display(table)

In [ ]:
#@title Playback helpers: play_action, play_goal (run this cell; double-click to view the code)
def play_moment(video, start, end, label=""):
    """Stream one interval of the video. Does not rerun Activity."""
    if pd.isna(start) or pd.isna(end) or end <= start:
        raise ValueError("This row has no valid time interval; choose another row.")
    stream_url = video.generate_stream(timeline=[(float(start), float(end))])
    print(f"{clock(start)}–{clock(end)}  {label}")
    print("Stream URL:", stream_url)
    return play_stream(stream_url)


def play_action(video, actions, row):
    """Play the action at this row of the action table."""
    action = actions.iloc[row]
    return play_moment(video, action["start"], action["end"], action["action"])


def play_goal(video, goals, row):
    """Play the full interval of the goal at this row of the goal table."""
    goal = goals.iloc[row]
    return play_moment(video, goal.get("start"), goal.get("end"), goal.get("instruction"))

<a id="video"></a>
## 3. Upload a video

VideoDB fetches the video from the URL directly; no local download is needed. Set `VIDEODB_VIDEO_URL` to try your own.

In [ ]:
VIDEO_URL = os.getenv(
    "VIDEODB_VIDEO_URL",
    "https://www.youtube.com/watch?v=wuRf25_zME0",  # First-person laundry folding
)

video = collection.upload(url=VIDEO_URL)

# To reuse a video already in your account, replace the upload above with:
# video = collection.get_video("m-...")

print("Video:", video.id)
video.play()

<a id="run"></a>
## 4. Run Activity

Activity prepares the frames and runs both analysis passes for you, using the server's default model. Pick a mode:

- **`express`** starts processing immediately.
- **`batch`** saves 50% on model inference costs, but can take longer.

Each run of this cell creates a new Understanding.

In [ ]:
MODE = "express"  # or "batch"

understanding = video.understand(
    analyzers=[{"type": "activity", "name": "activity", "config": {"mode": MODE}}],
)
print("Understanding ID:", understanding.id)

<a id="wait"></a>
## 5. Wait for results

If this times out, rerun this cell: it keeps waiting on the same Understanding.

In [ ]:
understanding.wait_until_complete(timeout=48 * 3600, poll_interval=60)

analyzer = understanding.get_analyzer("activity", refresh=True)
if not analyzer.is_successful:
    raise RuntimeError(f"Activity did not succeed: {analyzer.status}")

actions, goals, warnings = activity_tables(analyzer.get_output())
print("Status:", analyzer.status)

<a id="results"></a>
## 6. Explore the results

Start with the overview: each lane is a goal, each block an action. Hover a block for its text; a dashed red outline marks an action with an issue.

In [ ]:
show_summary(actions, goals, warnings)
show_timeline(actions)

### Actions

One row per action: when it happens, what is done, the goal it serves, and the model's confidence. Check rows with an `issue` before relying on them.

In [ ]:
show_actions(actions)

### Goals

A goal can span several actions. A `–` start or end means there were not enough supporting actions to place it.

In [ ]:
show_goals(goals)

<a id="play"></a>
## 7. Play a moment

Pass a row number from the tables above. The printed stream URL also works in your own player.

In [ ]:
play_action(video, actions, row=0)

In [ ]:
play_goal(video, goals, row=0)

<a id="next"></a>
## Next steps

| Goal | Notebook |
|---|---|
| Explore every analyzer | [Understanding guide map](README.md) |
| Manage runs, statuses and callbacks | [Outputs and operations](outputs-and-operations.ipynb) |
| Compose dependent analyzers | [Multi-analyzer pipelines](multi-analyzer-pipelines.ipynb) |